# Colab SSH Setup

Run this notebook in Colab to get SSH access. Then control everything from your local Cursor terminal.

**Steps:**
1. Run all cells below
2. Copy the SSH command printed at the end
3. Paste it in your local terminal (Cursor)
4. Run your scripts on Colab's GPU from home

In [ ]:
# Step 1: Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Step 2: Mount Google Drive (persists models/data across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Install dependencies
!pip install -q \
    "numpy>=1.26,<2" \
    diffusers==0.30.0 \
    transformers==4.44.0 \
    accelerate==0.33.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    datasets==2.20.0 \
    huggingface_hub==0.24.5 \
    controlnet_aux==0.0.9 \
    xformers \
    einops \
    tomesd \
    "imageio[ffmpeg]" \
    opencv-python-headless \
    gradio==4.42.0 \
    safetensors \
    sentencepiece \
    protobuf
print("Dependencies installed.")

In [ ]:
# Step 4: Clone the repo and set up project structure
import os

DRIVE_ROOT = "/content/drive/MyDrive/ImgGen"
PROJECT = "/content/ImgGen"

# Clone repo if not already there
if not os.path.exists(PROJECT):
    !git clone https://github.com/Mehul159/ImgGen.git /content/ImgGen
else:
    !cd /content/ImgGen && git pull

# Symlink heavy dirs to Google Drive
for folder in ["models", "lora_weights", "data"]:
    drive_path = os.path.join(DRIVE_ROOT, folder)
    local_path = os.path.join(PROJECT, folder)
    os.makedirs(drive_path, exist_ok=True)
    if os.path.isdir(local_path) and not os.path.islink(local_path):
        !rm -rf {local_path}
    if not os.path.exists(local_path):
        os.symlink(drive_path, local_path)
        print(f"  {folder}/ -> Drive")

os.makedirs(os.path.join(PROJECT, "outputs"), exist_ok=True)
print(f"\nProject ready at {PROJECT}")

In [ ]:
# Step 5: Set up SSH with cloudflared tunnel
import subprocess
import secrets
import time

# Set root password
password = secrets.token_urlsafe(12)
subprocess.run(['bash', '-c', f'echo "root:{password}" | chpasswd'], check=True)

# Configure SSH
subprocess.run(['bash', '-c', '''
apt-get -qq update && apt-get -qq install -y openssh-server > /dev/null 2>&1
mkdir -p /var/run/sshd
echo "PermitRootLogin yes" >> /etc/ssh/sshd_config
echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config
service ssh start
'''], check=True)

# Install cloudflared
subprocess.run(['bash', '-c', '''
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
rm cloudflared-linux-amd64.deb
'''], check=True)

# Start tunnel in background
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

time.sleep(8)

# Extract tunnel URL
import re
output = proc.stderr.read1(4096).decode() if hasattr(proc.stderr, 'read1') else ''
if not output:
    time.sleep(5)
    output = open('/root/.cloudflared/logs.txt', 'r').read() if os.path.exists('/root/.cloudflared/logs.txt') else ''

# Try reading stderr properly
import select
stderr_text = ""
for _ in range(10):
    if select.select([proc.stderr], [], [], 1)[0]:
        chunk = os.read(proc.stderr.fileno(), 8192).decode()
        stderr_text += chunk
    url_match = re.search(r'https://[\w-]+\.trycloudflare\.com', stderr_text)
    if url_match:
        break
    time.sleep(1)

url_match = re.search(r'https://[\w-]+\.trycloudflare\.com', stderr_text)

if url_match:
    tunnel_url = url_match.group(0)
    hostname = tunnel_url.replace('https://', '')
    print("="*60)
    print("SSH TUNNEL READY")
    print("="*60)
    print(f"\nPassword: {password}")
    print(f"\n--- Run these on your LOCAL machine ---\n")
    print(f"Step 1 (one-time): Add to ~/.ssh/config:\n")
    print(f"  Host colab")
    print(f"    HostName {hostname}")
    print(f"    User root")
    print(f"    ProxyCommand cloudflared access ssh --hostname %h")
    print(f"\nStep 2: Connect:\n")
    print(f"  ssh colab")
    print(f"\n--- OR one-liner (no config needed) ---\n")
    print(f"  ssh -o ProxyCommand='cloudflared access ssh --hostname {hostname}' root@{hostname}")
    print(f"\nPassword when prompted: {password}")
    print(f"\nThen: cd /content/ImgGen && python preprocess.py")
    print("="*60)
else:
    print("Could not detect tunnel URL. stderr output:")
    print(stderr_text[:2000])

In [ ]:
# Step 6: Keep alive — run this cell and leave it running
# Prevents Colab from disconnecting due to inactivity
import time
print("Keeping session alive... (leave this running)")
print("You can now SSH in from your local terminal.")
while True:
    time.sleep(60)
    print(".", end="", flush=True)